# Beyond-Erasure EXIT/GEXIT Diagnostics

This notebook runs the first mixed erasure plus Pauli component diagnostic. The main quantity is `H(C_X | Y,S)` or `H(C_Z | Y,S)`, where `Y` includes the erasure locations and the channel model. Rows with `list_runs > 0` are list-restricted MCMC estimates, not exact ML posterior entropies.

In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / 'experiments').exists():
    ROOT = ROOT.parent
PYTHON = ROOT / '.venv' / 'Scripts' / 'python.exe'
if not PYTHON.exists():
    PYTHON = Path(sys.executable)
OUT_DIR = ROOT / 'data' / 'experiments' / 'beyond_erasure'
CODE = ROOT / 'codes' / 'surface5_HxHzLxLz.npz'
ROOT, PYTHON, CODE


## Quick Heatmap

This is a smoke-sized run. It keeps the exact cutoff low so that non-erasure points use the list-restricted path quickly. Increase `--runs`, `--approx-samples`, and the grids for exploratory plots. For exact surface5 Pauli-only X-side samples, request `--max-exact-affine-dim 21` deliberately.

In [ ]:
cmd = [
    str(PYTHON),
    str(ROOT / 'experiments' / 'beyond_erasure_diagnostics.py'),
    '--code', str(CODE),
    '--component', 'x',
    '--runs', '5',
    '--p-erasure-grid', '0', '0.1', '0.2',
    '--p-error-grid', '0', '0.01', '0.03',
    '--max-exact-affine-dim', '12',
    '--approx-samples', '200',
    '--mcmc-burnin', '200',
    '--mcmc-thin', '1',
    '--plot',
]
subprocess.run(cmd, cwd=ROOT, check=True)


In [ ]:
json_path = OUT_DIR / 'surface5_HxHzLxLz_beyond_erasure_x.json'
csv_path = OUT_DIR / 'surface5_HxHzLxLz_beyond_erasure_x.csv'
result = json.loads(json_path.read_text())
rows = list(csv.DictReader(csv_path.open()))
result['code'], result['config'], rows[:3]


## Exact Single-Point Check

This can take tens of seconds on surface5 because it enumerates roughly `2^21` posterior states. Use it as a correctness check, not as the default heatmap mode.

In [ ]:
# subprocess.run([
#     str(PYTHON),
#     str(ROOT / 'experiments' / 'beyond_erasure_diagnostics.py'),
#     '--code', str(CODE),
#     '--component', 'x',
#     '--runs', '1',
#     '--p-erasure-grid', '0',
#     '--p-error-grid', '0.02',
#     '--max-exact-affine-dim', '21',
#     '--approx-samples', '0',
# ], cwd=ROOT, check=True)
